# Reproduction: TOPVIEWRS — Vision-Language Models as Top-View Spatial ReasonersLi, Zhang, Zhou, Collier, Korhonen, Vulic (arXiv:2406.02537v1, EMNLP 2024).## VERDICT: IT DOES NOT FITTwo independent blockers, both fatal to a *full* reproduction on the described Kaggle setup.### Blocker 1 — the mounted data is not this paper's data`prompt.txt` mounts a **GQA visual-CoT** annotation file (`gqa_cot_val.jsonl`, 9,855 records,fields `question / answer / full_answer / image / width / height / bboxs / reasoning / thought`)and the **GQA images** dataset.TOPVIEWRS is a different benchmark: <cite>"11,384 multiple-choice questions with eitherrealistic or semantic top-view map"</cite> (Abstract), <cite>"we select a total of 7 scenes"</cite>from Matterport3D (App. B.1). It has no bounding boxes, no free-form answers, and no GQA images.There is no path from the mounted files to this paper's numbers.Per your own rule (`<feasibility_first>`): *"If data is missing, that means a wrong path... Stop andprint what is actually mounted."* **Cell 1 audits both mounts exactly as you specified, then raises.**The notebook does not substitute GQA for TOPVIEWRS anywhere.The TOPVIEWRS eval data is fetched separately from the authors' HuggingFace release.### Blocker 2 — the public TOPVIEWRS release is a subset, and the model roster does not fit 2xT4The HF card for `chengzu/topviewrs` states only part of the benchmark (2 of the 7 scenes) isreleased, to avoid contamination; full access is by email to the authors. And of the paper's 10models (§5 *Models and Implementation*), Idefics-80B, LLaVA-Next-34B, GPT-4V and Gemini cannot runhere.---## Deviation tableEvery row below changes the numbers relative to the paper. Nothing was shrunk silently.| # | Item | Paper's value | This run's value | Effect on results ||---|------|---------------|------------------|-------------------|| D1 | Eval corpus | 11,384 MCQs; 5,539 realistic / 5,845 semantic (§4 *Dataset Statistics*) | Whatever `chengzu/topviewrs` ships; counts checked against those three numbers at load time | If the counts match, corpus size is not a deviation and `fidelity` is set to `full reproduction`. If they don't, `n_eval` per row still comes from the loaded data, never from the paper. || D2 | Input mounts | TOPVIEWRS realistic + semantic maps | `/kaggle/input/...` holds GQA CoT + GQA images | Mounts are audited and reported, then rejected. Zero mounted records enter evaluation. || D3 | Models: Idefics-80B | Evaluated (§5, Table 1) | **Dropped.** ~160 GB fp16 vs 32 GB total VRAM | No Idefics-80B row; the paper's "larger models are not always better" claim cannot be checked on this family. || D4 | Models: LLaVA-Next-34B | Evaluated (§5, Table 1) | **Dropped.** ~68 GB fp16 vs 32 GB | No 34B row; 34B is the best open model on 5 of 8 task/map cells in Table 1, so the open-source ceiling here is lower. || D5 | Models: GPT-4V, Gemini | `GPT-4-turbo-2024-04-09`, `gemini-pro-vision 1.0` (fn. 5) | **Dropped by default.** Both snapshots are retired; needs paid keys | Table 1's two best columns and all of Table 3 (CoT) are unreproducible as specified. || D6 | Numeric precision | NOT SPECIFIED | fp16 (`torch.float16`); T4 has no bf16 | Small drift vs any bf16/fp32 reference. XComposer2 and Qwen-VL ship bf16 defaults upstream; fp16 may destabilise them. || D7 | Harness | VLMEvalKit (§5) | Plain `transformers` loop; Table 10 generation params copied verbatim | Prompt assembly and decoding follow the paper's tables, but VLMEvalKit's own answer-extraction heuristics are not replicated (see A5). || D8 | Human evaluation | 4 participants, 60 items, Fleiss kappa 0.747 (§5.2, Table 2) | **Not run** | Table 2 and the ">50% gap to human" headline are not reproduced. || D9 | CoT ablation | GPT-4V and Gemini only, Static Spatial Reasoning (§5.2, Table 3) | Optional, on the open models that do fit | If enabled, the delta is measured on different models than the paper's, so it does not validate the reported +5.82%. Off by default. || D10 | Session budget | NOT SPECIFIED | 12 h/session; ~11k items x ~2-4 s/item means roughly one model per session | Predictions are cached to JSONL and resumed, so a multi-model table needs several sessions. || D13 | Scene set | 7 MP3D scenes named in App. B.1 | Whatever the release ships; the discovery cell prints the scene list and flags extras | Extra scenes change the sub-task mix relative to Table 9. Recorded in `deviations_runtime.csv`. || D12 | Library versions | NOT SPECIFIED (paper names only VLMEvalKit) | Kaggle image versions, minimum bounds enforced, numpy frozen | Kernel-level differences in `transformers` image preprocessing can shift LLaVA-Next patching and therefore scores. `ENV_VERSIONS` is recorded in `deviations_runtime.csv`. || D11 | Cell-1 grounding-accuracy check | Not a TOPVIEWRS quantity — the paper has no box-prediction task | Reported as **N/A** | Your `<cell_1_checks>` full-frame/IoU split applies to the GQA file only. Box statistics are still printed in the mount audit. |---

# Part 1 — Paper specification## 1.1 Goal and proposed "architecture" (3-5 sentences)The paper is a **benchmark and evaluation study**, not a new model: there is no trained architectureto reproduce. It asks whether VLMs can read and reason over *top-view* (bird's-eye) maps of indoorscenes, a perspective prior spatial-reasoning work ignored in favour of first-person views (§1).The authors build TOPVIEWRS, <cite>"11,384 multiple-choice questions"</cite> over two visualformats — photo-realistic orthographic renders and semantic colour-box maps — organised into 4 tasksof increasing complexity spanning 9 sub-tasks (§3). The "system under test" is therefore afrozen, zero-shot VLM plus a task-specific prompt template; the contribution is the controlledtask decomposition that <cite>"disentangle[s] different abilities"</cite> (Abstract).The evaluated pipeline is: map image + templated prompt -> VLM -> single option letter -> EM/PM scoring.## 1.2 Module table| Module | Function | Input shape | Output shape | Section ||---|---|---|---|---|| Map renderer (realistic) | Orthographic camera shot over MP3D mesh in MeshLab | 3D mesh (7 scenes) | RGB image `H x W x 3` | App. B.1 || Map renderer (semantic) | Habitat `get_topdown_map` + coloured boxes drawn low-object-first | 3D mesh + MP3D annotations | RGB image `H x W x 3` | App. B.1 || Question generator | 15 templates -> MCQ with 1 gold + 3 distractors, options shuffled | scene annotations | `question:str`, `choices:list[4]`, `labels:list[1]` | App. B.2 || Human alignment | keep / modify / correct / discard each item | raw MCQ | verified MCQ | §4, App. B.3 || Colour-object mapping selector | emit only RGB->object pairs present in this map | semantic image | `<MAPPING>` text block | §5 *Prompts*; Table 4 || Prompt assembler | fill `<QUESTION>`, `<OPTIONS>`, `<MAPPING>`, `<TASK-SPECIFIC INSTRUCTION>` | MCQ + map type + task | prompt string | Tables 11-14 || VLM (frozen, zero-shot) | image + prompt -> text | image + prompt | <= 20 new tokens (Table 10) | §5 || Answer extractor | generated text -> option index | text | index in `{0,1,2,3}` | NOT SPECIFIED || Scorer | EM and PM | pred index, gold index, option texts | two scalars | §5 *Evaluation Measures* |**Note on shapes:** the paper never states map resolution or a resize policy. GPT-4V is the onlymodel with an image size in the paper (`img_size 512`, `img_detail low`, Table 10). All otherinput resolutions are whatever each checkpoint's own processor does — **NOT SPECIFIED**.## 1.3 Data flow, in order1. Select map type `M in {realistic, semantic}` and task split (one of 4).2. Load record: `question`, `choices[4]`, `labels`, `map_path`, `question_ability`, (`reference_path` for DSR).3. Open the map image at `map_path`.4. If semantic: scan the image for unique RGB values, keep those in the Table-4 palette, build `<MAPPING>`.5. Render `<OPTIONS>` as `A. <c0>; B. <c1>; C. <c2>; D. <c3>` (App. C.2).6. If task is Dynamic Spatial Reasoning and sub-task is Dynamic Action Counting, insert the turn-counting `<TASK-SPECIFIC INSTRUCTION>` (App. C.2); otherwise insert the empty string.7. Select the template: Table 11 (realistic), Table 12 (semantic), Table 13/14 for CoT.8. Generate with that model's Table-10 parameters.9. Extract the option letter -> predicted index.10. Score EM against the gold index; score PM over word sets of the option texts.11. Aggregate per (model, map type, task) for Table 1 and per sub-task for Table 15.## 1.4 Reproduction table| Item | Value | Where ||---|---|---|| Dataset | TOPVIEWRS, 11,384 MCQs, from Matterport3D (7 scenes: 17DRP5sb8fy, 2azQ1b91cZZ, 2t7WUuJeko7, 5LpN3gDmAk7, EU6Fwq7SyZv, 8WUmhLawc2A, i5noydFURQK) | §4; App. B.1 || Split | Single evaluation split. 5,539 realistic / 5,845 semantic. Sub-task sizes in Table 9. No train/val/test — zero-shot only | §4; Table 9 || Preprocessing (image) | Realistic: orthographic render. Semantic: coloured boxes, lower objects drawn first, full-floor maps cropped into rooms via Habitat region boundaries. Resize/normalisation at eval time: **NOT SPECIFIED** | App. B.1 || Preprocessing (text) | Fill templates in Tables 11-14; semantic maps additionally get the present-colours-only RGB->object mapping | §5 *Prompts*; App. C.2 || Feature selection | Not applicable — no feature-selection stage in the paper | — || Model per stage | Single stage, frozen VLM. Idefics 9B & 80B; LLaVA-Next vicuna-7B, mistral-7B, vicuna-13B, 34B; InternLM-XComposer2 7B; Qwen-VL 7B; GPT-4V (`gpt-4-turbo-2024-04-09`); Gemini (`gemini-pro-vision 1.0`). Exact HF checkpoint ids: **NOT SPECIFIED** | §5; fn. 5 || Loss | **NOT SPECIFIED** — no training is performed anywhere in the paper | — || Optimizer | **NOT SPECIFIED** — none | — || Learning rate / schedule | **NOT SPECIFIED** — none | — || Batch size | **NOT SPECIFIED** | — || Epochs | **NOT SPECIFIED** — zero-shot inference, 0 epochs | §5 || Regularization | **NOT SPECIFIED** — none | — || Decoding | Table 10: Idefics `max_new_tokens=20`; LLaVA-Next `temperature=0, num_beams=1, max_new_tokens=20, do_sample=False, top_p=None`; XComposer2 `temperature=1, beams=5, max_token=20, repetition_penalty=1, do_sample=False`; Qwen-VL `max_new_tokens=20`; GPT-4V `temperature=0, max_tokens=1024, img_size=512, img_detail=low`; Gemini `temperature=0, max_tokens=1024` | Table 10 || Metrics | Exact Match (predicted option index == label index) and Partial Match, `PM = \|labels ∩ predictions\| / max(\|labels\|, \|predictions\|)`, over word spans of the option text | §5 *Evaluation Measures* || Random baseline | Present in Figs. 1 and 3; its numeric value is **NOT SPECIFIED** in text (4 options, single correct answer) | §3 fn. 4; Fig. 3 || Ablation 1 | Chain-of-Thought on Static Spatial Reasoning, GPT-4V + Gemini, instruction <cite>"You should first localize the entity and then answer the question"</cite> | §5.2; Table 3; Tables 13-14 || Ablation 2 | Realistic vs semantic map type (run for all models) | §5.1; Table 1 || Ablation 3 | Sub-task breakdown, 9 sub-tasks | Fig. 3; Table 15 || Ablation 4 | Human vs GPT-4V on 60 realistic-map items, 4 annotators, Fleiss kappa 0.747 | §5.2; Table 2 || Seed | **NOT SPECIFIED** | — || Hardware / runtime | **NOT SPECIFIED** | — |## 1.5 Assumptions tableEverything below is a choice this notebook made because the paper is silent. None of it isattributed to the paper. Each appears in code as `# ASSUMPTION:`.| ID | Gap in paper | Value used | Why ||---|---|---|---|| A1 | Exact HF checkpoints for the named models | `llava-hf/llava-v1.6-{vicuna-7b,mistral-7b,vicuna-13b}-hf`, `HuggingFaceM4/idefics-9b-instruct`, `internlm/internlm-xcomposer2-vl-7b`, `Qwen/Qwen-VL-Chat` | These are the ids VLMEvalKit registers under the names in §5. Idefics base-vs-instruct is not stated. || A2 | Numeric precision | `torch.float16` | Only fp16 is available on T4 (see D6). || A3 | Batch size | 1 | Variable-resolution image inputs; no padding policy given. || A4 | Seed | 0 for python / numpy / torch | Decoding is greedy for most models, so the seed mostly affects load order. || A5 | Answer extraction from generated text | First standalone `A`/`B`/`C`/`D`; else text after `The answer is`; else normalized exact match against an option string; else `UNPARSED` | The paper reports that some models <cite>"fail to respond to instructions"</cite> (§5.1) but never gives the parser. || A6 | Normalization for PM word sets | lowercase, strip punctuation, split on whitespace and `_`, drop empties | PM is defined over "text spans (or words)"; the tokenizer is not given. || A7 | How `<MAPPING>` colours are detected | Exact-RGB scan of the semantic PNG against the 40-row Table 4 palette; anti-aliased near-colours ignored | §5 says only present colours are shown, but not how presence is computed. || A8 | Ordering of `<MAPPING>` lines | Descending pixel count | Not specified; deterministic so runs are comparable to each other. || A9 | `Prec/Recall/F1/FPR/FNR` averaging | macro over the 4 letter classes, one-vs-rest; `UNPARSED` counts as a wrong prediction | The paper reports only EM and PM. These columns exist because your spec asks for them. || A10 | Which sub-tasks get the `<TASK-SPECIFIC INSTRUCTION>` | Only `dynamic_action_counting` | App. C.2 gives the instruction for that sub-task and says the string is empty otherwise. || A11 | Library versions | Minimum bounds only (`transformers>=4.39`, `accelerate>=0.26`); numpy and pyarrow frozen at whatever the image ships, via a pip constraint file | Hard pins pull numpy below 2.0 mid-kernel and every loaded C extension dies with `numpy.dtype size changed`. The env cell verifies numpy did not move and raises if it did. || A12 | How the benchmark is read when `datasets>=4` has removed script loading | Three loaders in order: the documented `load_dataset` script API; a line-for-line port of the authors' `_generate_examples` (reading `released_<map_type>_datasets.json[task_split]`, `map_path = join(image_save_dir, item['rgb_map' or 'semantic_map'])`); the authors' script imported and driven directly | The card documents only the script API. Both loaders read the authors' own files; if both fail the cell raises and prints the real listing. |---

# Part 2 — Notebook## CONFIG

In [ ]:
# =============================== CONFIG =====================================# Every path and hyperparameter lives here. Nothing below this cell hardcodes a value.import os, sys, json, math, time, random, string, hashlib, platformfrom dataclasses import dataclass, fieldfrom typing import Any, Dict, List, Optional, Tuple# ---- paths from <my_setup> (these are the WRONG-PAPER mounts; audited in Cell 1) ----GQA_ANNOT_PATH = "/kaggle/input/notebooks/khoangoo/test-dataset-visual-cot/visual-cot/cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl"GQA_IMAGE_ROOT = "/kaggle/input/datasets/lyte69/gqa-images/images"GQA_EXPECTED_RECORDS = 9855GQA_EXPECTED_IMAGES = 5422FULL_FRAME_AREA_FRAC = 0.8   # ASSUMPTION: prompt says "covers most of the frame" without a number; using 0.8# ---- TOPVIEWRS eval data (the paper's actual data) ----# If you have obtained the FULL benchmark from the authors (cl917@cam.ac.uk) and mounted it,# point TOPVIEWRS_LOCAL_DIR at it. Otherwise the public 2-scene subset is downloaded (deviation D1).TOPVIEWRS_LOCAL_DIR = NoneTOPVIEWRS_HF_REPO   = "chengzu/topviewrs"IMAGE_SAVE_DIR      = "/kaggle/working/topviewrs_images"OUT_DIR   = "/kaggle/working"CACHE_DIR = "/kaggle/working/pred_cache"RESULTS_CSV = os.path.join(OUT_DIR, "results.csv")SUBTASK_CSV = os.path.join(OUT_DIR, "results_subtask.csv")   # analogue of paper Table 15COT_CSV     = os.path.join(OUT_DIR, "results_cot.csv")       # analogue of paper Table 3SEED = 0            # ASSUMPTION: paper does not specify a seed; using 0# ---- task grid, paper Sec. 3 / Table 9 ----MAP_TYPES   = ["realistic", "semantic"]TASK_SPLITS = ["top_view_recognition", "top_view_localization",               "static_spatial_reasoning", "dynamic_spatial_reasoning"]TASK_DISPLAY = {    "top_view_recognition":      "Top-View Recognition",    "top_view_localization":     "Top-View Localization",    "static_spatial_reasoning":  "Static Spatial Reasoning",    "dynamic_spatial_reasoning": "Dynamic Spatial Reasoning",}# Any value other than None here creates a subset run -> a new deviation row is written# automatically into deviations_runtime.csv. A subset score is NOT the paper's score.MAX_ITEMS_PER_TASK: Optional[int] = None# ---- model registry -------------------------------------------------------# hf_id values are ASSUMPTION A1: the paper names models but no checkpoint ids.# gen kwargs are copied verbatim from paper Table 10.MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {    "Idefics-9B": {        "hf_id": "HuggingFaceM4/idefics-9b-instruct",   # ASSUMPTION: paper does not specify base vs instruct        "family": "idefics",        "gen": {"max_new_tokens": 20},                   # Table 10        "paper_row": "Idefics 9B",    },    "LLaVANext-vicuna-7B": {        "hf_id": "llava-hf/llava-v1.6-vicuna-7b-hf",        "family": "llava_next",        "gen": {"temperature": 0.0, "num_beams": 1, "max_new_tokens": 20,                "do_sample": False, "top_p": None},      # Table 10        "paper_row": "LLaVANext vicuna 7B",    },    "LLaVANext-mistral-7B": {        "hf_id": "llava-hf/llava-v1.6-mistral-7b-hf",        "family": "llava_next",        "gen": {"temperature": 0.0, "num_beams": 1, "max_new_tokens": 20,                "do_sample": False, "top_p": None},        "paper_row": "LLaVANext mistral 7B",    },    "LLaVANext-vicuna-13B": {        "hf_id": "llava-hf/llava-v1.6-vicuna-13b-hf",        "family": "llava_next",        "gen": {"temperature": 0.0, "num_beams": 1, "max_new_tokens": 20,                "do_sample": False, "top_p": None},        "paper_row": "LLaVANext vicuna 13B",    },    "InternLM-XComposer2": {        "hf_id": "internlm/internlm-xcomposer2-vl-7b",        "family": "xcomposer2",        "gen": {"temperature": 1.0, "num_beams": 5, "max_new_tokens": 20,                "repetition_penalty": 1.0, "do_sample": False},   # Table 10 ("beams", "max_token")        "paper_row": "XComposer2 7B",    },    "Qwen-VL": {        "hf_id": "Qwen/Qwen-VL-Chat",                    # ASSUMPTION: paper says "Qwen-VL (7B)" only        "family": "qwen_vl",        "gen": {"max_new_tokens": 20},                   # Table 10        "paper_row": "Qwen-VL 7B",    },}# Dropped for VRAM (deviations D3, D4). Listed so the gap is explicit, not silent.MODELS_DROPPED_VRAM = {    "Idefics-80B":      "HuggingFaceM4/idefics-80b-instruct",    "LLaVANext-34B":    "llava-hf/llava-v1.6-34b-hf",}# Dropped for retired snapshots / paid keys (deviation D5).MODELS_DROPPED_API = {    "GPT-4V": "gpt-4-turbo-2024-04-09",       # paper footnote 5    "Gemini": "gemini-pro-vision-1.0",        # paper footnote 5}# Run one model per 12h session (deviation D10). Predictions are cached and resumable.MODELS_TO_RUN: List[str] = ["LLaVANext-vicuna-7B"]# Chain-of-Thought ablation, paper Sec. 5.2 / Table 3. Paper ran it on GPT-4V and Gemini only.# Enabling it here measures a DIFFERENT quantity (deviation D9).RUN_COT = FalseCOT_TASK = "static_spatial_reasoning"          # Table 3 covers this task onlyDTYPE_STR = "float16"       # ASSUMPTION A2BATCH_SIZE = 1              # ASSUMPTION A3# Kaggle's image already ships a working numpy/torch/CUDA stack. Hard-pinning transformers or# datasets drags numpy below 2.0, which shatters every pre-imported C extension in the live# kernel. So: state MINIMUM versions, constrain numpy to whatever is already installed, and# install only what is actually missing or too old.MIN_TRANSFORMERS = (4, 39, 0)   # first release with LlavaNextForConditionalGenerationMIN_ACCELERATE   = (0, 26, 0)   # device_map='auto' + max_memoryOPTIONAL_PKGS    = {"sentencepiece": None, "einops": None, "timm": None}PIP_CONSTRAINTS_PATH = os.path.join(OUT_DIR, "pip-constraints.txt")# ---- Prompt templates, verbatim from paper Appendix C.2 -------------------PROMPT_REALISTIC_STATIC = (    "This is a top-view map of a room. Please respond to the question below by selecting one choice "    "from a list of available options provided. Your response should only include the letter of the "    "chosen option (A, B, C, or D) with no additional explanation.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer:")   # Table 11PROMPT_REALISTIC_DYNAMIC = (    "This is a top-view map of a room with the navigation path. The path starts from the green "    "triangle (RGB [0, 255, 0]) and ends at the red star (RGB [255, 0, 0]). The direction of the "    "path is denoted by a series of yellow arrows (RGB [255, 255, 0]), with intermediate points "    "highlighted in RGB [25, 255, 255]. <TASK-SPECIFIC INSTRUCTION> Please respond to the question "    "below by selecting one choice from a list of available options provided. Your response should "    "only include the letter of the chosen option (A, B, C, or D) with no additional explanation.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer:")   # Table 11PROMPT_SEMANTIC_STATIC = (    "This is a semantic top-view map of a room. Various objects are depicted by colored bounding "    "boxes, each with its corresponding color, and there may be instances of overlap between them. "    "Below are the RGB color codes associated with each object, presented in the format RGB -> "    "Object:\n"    "<MAPPING>\n"    "Please respond to the question below by selecting one choice from a list of available options "    "provided. Your response should only include the letter of the chosen option (A, B, C, or D) "    "with no additional explanation.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer:")   # Table 12PROMPT_SEMANTIC_DYNAMIC = (    "This is a semantic top-view map of a room with the navigation path. In the semantic map, "    "various objects are depicted by colored bounding boxes, each with its corresponding color, and "    "there may be instances of overlap between them. The navigation path starts from the green "    "triangle (RGB [0, 255, 0]) and ends at the red star (RGB [255, 0, 0]). The direction of the "    "path is denoted by a series of yellow arrows (RGB [255, 255, 0]), with intermediate points "    "highlighted in RGB [25, 255, 255]. Below are the RGB color codes associated with each object "    "and symbol, presented in the format RGB -> Object:\n"    "<MAPPING>\n"    "<TASK-SPECIFIC INSTRUCTION> Please respond to the question below by selecting one choice from a "    "list of available options provided. Your response should only include the letter of the chosen "    "option (A, B, C, or D) with no additional explanation.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer:")   # Table 12PROMPT_COT_REALISTIC = (    "This is a top-view map of a room. Please respond to the question below by selecting one choice "    "from a list of available options provided. You should explain your reasoning step-by-step by "    "first localizing the entities and then reasoning over the question based on the locations. You "    "should conclude your chosen option (A, B, C, or D) starting with 'The answer is '.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer: Let's think step by step.")   # Table 13PROMPT_COT_SEMANTIC = (    "This is a semantic top-view map of a room. Various objects are depicted by colored bounding "    "boxes, each with its corresponding color, and there may be instances of overlap between them. "    "Below are the RGB color codes associated with each object, presented in the format RGB -> "    "Object:\n"    "<MAPPING>\n"    "Please respond to the question below by selecting one choice from a list of available options "    "provided. You should explain your reasoning step-by-step by first localizing the entities and "    "then reasoning over the question based on the locations. You should conclude your chosen option "    "(A, B, C, or D) starting with 'The answer is '.\n"    "Question: <QUESTION>\n"    "Options: <OPTIONS>;\n"    "Answer: Let's think step by step.")   # Table 14TASK_SPECIFIC_INSTRUCTION = (    "Suppose you are a navigation agent tracing the path. Your job is to assess whether there's a "    "turn at each intermediate point and sum up the total turns for the final outcome.")   # Appendix C.2, for sub-task Dynamic Action Counting# ASSUMPTION A10: applied to this sub-task only; empty string elsewhere (App. C.2).INSTRUCTION_SUBTASKS = {"dynamic_action_counting"}# CoT max_tokens: paper's CoT models used max_tokens 1024 (Table 10, GPT-4V / Gemini rows).COT_MAX_NEW_TOKENS = 1024# ---- Table 4: RGB -> label palette, verbatim from paper Appendix B.1 ------RGB_TO_LABEL = {    (31, 119, 180): "void",            (174, 199, 232): "wall",    (255, 127, 14): "floor",           (255, 187, 120): "chair",    (44, 160, 44): "door",             (152, 223, 138): "table",    (214, 39, 40): "picture",          (255, 152, 150): "cabinet",    (148, 103, 189): "cushion",        (197, 176, 213): "window",    (140, 86, 75): "sofa",             (196, 156, 148): "bed",    (227, 119, 194): "curtain",        (247, 182, 210): "chest_of_drawers",    (51, 105, 30): "plant",            (199, 199, 199): "sink",    (188, 189, 34): "stairs",          (219, 219, 141): "ceiling",    (23, 190, 207): "toilet",          (158, 218, 229): "stool",    (57, 59, 121): "towel",            (82, 84, 163): "mirror",    (107, 110, 207): "tv_monitor",     (156, 158, 222): "shower",    (99, 121, 57): "column",           (140, 162, 82): "bathtub",    (181, 207, 107): "counter",        (206, 219, 156): "fireplace",    (140, 109, 49): "lighting",        (189, 158, 57): "beam",    (231, 186, 82): "railing",         (231, 203, 148): "shelving",    (132, 60, 57): "blinds",           (173, 73, 74): "gym_equipment",    (214, 97, 107): "seating",         (231, 150, 156): "board_panel",    (123, 65, 115): "furniture",       (165, 81, 148): "appliances",    (206, 109, 189): "clothes",        (222, 158, 214): "objects",}# Navigation-path symbol colours, from the Table 12 prompt text itself.PATH_SYMBOL_COLORS = {    (0, 255, 0):    "start point of the navigation path (green triangle)",    (255, 0, 0):    "end point of the navigation path (red star)",    (255, 255, 0):  "direction arrow of the navigation path",    (25, 255, 255): "intermediate point of the navigation path",}OPTION_LETTERS = ["A", "B", "C", "D"]os.makedirs(OUT_DIR, exist_ok=True)os.makedirs(CACHE_DIR, exist_ok=True)os.makedirs(IMAGE_SAVE_DIR, exist_ok=True)print("CONFIG loaded.")

## Environment, pins, seeding

In [ ]:
# ---- environment: install only what is missing, never move numpy (ASSUMPTION A11) ----# Do NOT import numpy or torch in this cell. If pip replaces numpy underneath an already# imported extension module the kernel dies with#   ValueError: numpy.dtype size changed, may indicate binary incompatibility# and no later cell can recover, because a notebook cannot restart itself mid-run.import subprocess, sys, osimport importlib.metadata as ilmdWATCH = ["numpy", "torch", "transformers", "accelerate", "datasets", "huggingface-hub",         "pillow", "pandas", "scikit-learn", "pyarrow", "sentencepiece", "einops", "timm"]def ver(pkg):    try:        return ilmd.version(pkg)    except ilmd.PackageNotFoundError:        return Nonedef vtuple(s):    if s is None:        return None    out = []    for part in s.split("+")[0].split("."):        if part.isdigit():            out.append(int(part))        else:            break    return tuple(out)BEFORE = {p: ver(p) for p in WATCH}print("installed versions BEFORE:")for k in WATCH:    print("  %-16s %s" % (k, BEFORE[k]))if BEFORE["numpy"] is None or BEFORE["torch"] is None:    raise RuntimeError("numpy and torch must already be present in the image; they are not. "                       "Refusing to build a fresh scientific stack inside a live kernel.")# Freeze numpy (and pyarrow, which datasets likes to move) at whatever the image ships.with open(PIP_CONSTRAINTS_PATH, "w") as f:    f.write("numpy==%s\n" % BEFORE["numpy"])    if BEFORE["pyarrow"]:        f.write("pyarrow==%s\n" % BEFORE["pyarrow"])need = []if vtuple(BEFORE["transformers"]) is None or vtuple(BEFORE["transformers"]) < MIN_TRANSFORMERS:    need.append("transformers>=%d.%d.%d,<5" % MIN_TRANSFORMERS)if vtuple(BEFORE["accelerate"]) is None or vtuple(BEFORE["accelerate"]) < MIN_ACCELERATE:    need.append("accelerate>=%d.%d.%d" % MIN_ACCELERATE)for pkg in OPTIONAL_PKGS:    if BEFORE[pkg] is None:        need.append(pkg)if need:    cmd = [sys.executable, "-m", "pip", "install", "-q",           "--constraint", PIP_CONSTRAINTS_PATH,           "--upgrade-strategy", "only-if-needed"] + need    print("\n" + " ".join(cmd))    subprocess.run(cmd, check=True)   # check=True: a failed install raises, it does not fall throughelse:    print("\nnothing to install; the image already satisfies every minimum.")AFTER = {p: ver(p) for p in WATCH}moved = {k: (BEFORE[k], AFTER[k]) for k in WATCH if BEFORE[k] != AFTER[k]}print("\npackages that changed:", moved if moved else "none")if AFTER["numpy"] != BEFORE["numpy"]:    raise RuntimeError(        "numpy moved %s -> %s during install. Every C extension already loaded in this kernel "        "is now binary-incompatible and the session cannot be repaired in place. Restart the "        "session and re-run; if it happens again, remove the offending entry from `need`."        % (BEFORE["numpy"], AFTER["numpy"]))# Recorded so the deviation table reports the real environment rather than a wished-for one.ENV_VERSIONS = dict(AFTER)

In [ ]:
import random, numpy as np, torchdef set_seeds(seed: int):    random.seed(seed)    np.random.seed(seed)    torch.manual_seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)set_seeds(SEED)N_GPU = torch.cuda.device_count()DTYPE = getattr(torch, DTYPE_STR)DEVICE = "cuda" if N_GPU > 0 else "cpu"if DEVICE == "cpu":    DTYPE = torch.float32     # fp16 matmul is not supported on CPUprint("python           :", platform.python_version())print("torch            :", torch.__version__)print("cuda available   :", torch.cuda.is_available())print("n_gpu            :", N_GPU)for i in range(N_GPU):    p = torch.cuda.get_device_properties(i)    print("  gpu[%d]          : %s, %.1f GB, bf16=%s" % (i, p.name, p.total_memory/1e9,          torch.cuda.is_bf16_supported()))print("dtype            :", DTYPE)print()print("NONDETERMINISM that seeding does not remove:")print(" - cuDNN/cuBLAS kernel selection and fp16 reduction order can change logits at the")print("   1e-3 level; with greedy decoding this can flip a near-tie between two option letters.")print(" - XComposer2 uses beam search (Table 10, beams=5); beam tie-breaking is order dependent.")print(" - device_map='auto' can shard differently on a different GPU topology, which changes")print("   the order of fp16 accumulations.")print(" - HuggingFace checkpoint revisions are not pinned by the paper, so upstream reuploads")print("   would change results.")

## CELL 1 — setup checksRuns the audit you specified in `<cell_1_checks>` against the mounted GQA files, prints the result,then **stops**, because those files belong to a different benchmark (deviation D2). Nothing heregenerates substitute images.

In [ ]:
# ============================ CELL 1: SETUP CHECKS ==========================import os, jsonfrom PIL import Imageclass SetupError(RuntimeError):    passprint("=" * 78)print("MOUNT AUDIT")print("=" * 78)def show_tree(root, max_entries=40):    if not os.path.exists(root):        print("  MISSING:", root)        return    n = 0    for dirpath, dirnames, filenames in os.walk(root):        depth = dirpath[len(root):].count(os.sep)        if depth > 3:            dirnames[:] = []            continue        print("  " + "  " * depth + (os.path.basename(dirpath) or root), "(%d files)" % len(filenames))        n += 1        if n > max_entries:            print("  ... truncated")            returnprint("\n-- what is actually mounted under /kaggle/input --")show_tree("/kaggle/input")print("\nGQA_ANNOT_PATH :", GQA_ANNOT_PATH, "| exists:", os.path.exists(GQA_ANNOT_PATH))print("GQA_IMAGE_ROOT :", GQA_IMAGE_ROOT, "| exists:", os.path.exists(GQA_IMAGE_ROOT))gqa_audit = {}if os.path.exists(GQA_ANNOT_PATH) and os.path.isdir(GQA_IMAGE_ROOT):    n_files = len(os.listdir(GQA_IMAGE_ROOT))    print("files in IMAGE_ROOT:", n_files)    records = []    with open(GQA_ANNOT_PATH) as f:        for line in f:            line = line.strip()            if line:                records.append(json.loads(line))    print("records read       :", len(records), "(expected %d)" % GQA_EXPECTED_RECORDS)    uniq_imgs = sorted({r["image"] for r in records})    print("unique images      :", len(uniq_imgs), "(expected %d)" % GQA_EXPECTED_IMAGES)    missing = [fn for fn in uniq_imgs if not os.path.exists(os.path.join(GQA_IMAGE_ROOT, fn))]    print("images resolved    :", len(uniq_imgs) - len(missing), "/", len(uniq_imgs))    if missing:        print("FIRST 20 MISSING FILENAMES:")        for fn in missing[:20]:            print("   ", fn)        raise SetupError(            "%d of %d unique images do not resolve under IMAGE_ROOT. Fix the path; do not "            "substitute images." % (len(missing), len(uniq_imgs))        )    # real size vs declared width/height -> drop mismatches (bboxs are pixel coords)    real_size = {}    for fn in uniq_imgs:        with Image.open(os.path.join(GQA_IMAGE_ROOT, fn)) as im:            real_size[fn] = im.size          # header read only, no decode    kept, dropped = [], 0    for r in records:        w, h = real_size[r["image"]]        if int(r["width"]) == w and int(r["height"]) == h:            kept.append(r)        else:            dropped += 1    print("dropped (size mismatch):", dropped, "of", len(records))    # clamp boxes    clamped = 0    full_frame = 0    for r in kept:        w, h = int(r["width"]), int(r["height"])        new_boxes = []        for b in r.get("bboxs", []):            x1, y1, x2, y2 = b            cx1, cy1 = max(0.0, min(float(x1), w)), max(0.0, min(float(y1), h))            cx2, cy2 = max(0.0, min(float(x2), w)), max(0.0, min(float(y2), h))            if (cx1, cy1, cx2, cy2) != (float(x1), float(y1), float(x2), float(y2)):                clamped += 1            new_boxes.append([cx1, cy1, cx2, cy2])            if w * h > 0 and abs(cx2 - cx1) * abs(cy2 - cy1) >= FULL_FRAME_AREA_FRAC * w * h:                full_frame += 1        r["bboxs"] = new_boxes    print("boxes clamped          :", clamped)    print("boxes covering >= %.0f%% of frame: %d" % (FULL_FRAME_AREA_FRAC * 100, full_frame))    gqa_audit = {"records": len(records), "kept": len(kept), "dropped": dropped,                 "clamped": clamped, "full_frame_boxes": full_frame,                 "unique_images": len(uniq_imgs), "files_in_image_root": n_files}else:    print("The GQA mounts are not present at the configured paths.")print()print("Grounding accuracy with / without full-frame boxes: N/A.")print("  TOPVIEWRS has no box-prediction sub-task (paper Sec. 3 defines all 9 sub-tasks as")print("  4-way multiple choice), so there is no IoU to report. See deviation D11.")

In [ ]:
# ---------------------- HARD STOP: wrong benchmark mounted ------------------# The mounted annotations are GQA visual-CoT. The paper under reproduction is TOPVIEWRS.# Fields present in the mount: question/answer/full_answer/image/width/height/bboxs/reasoning/thought# Fields TOPVIEWRS needs:      question/choices/labels/map_path/question_ability/scene_id# There is no mapping between them. Per <feasibility_first>, we stop instead of substituting.TOPVIEWRS_FIELDS = {"question", "choices", "labels", "map_path", "question_ability", "scene_id"}def mount_provides_topviewrs() -> bool:    if TOPVIEWRS_LOCAL_DIR and os.path.isdir(TOPVIEWRS_LOCAL_DIR):        return True    if not os.path.exists(GQA_ANNOT_PATH):        return False    with open(GQA_ANNOT_PATH) as f:        first = json.loads(f.readline())    return TOPVIEWRS_FIELDS.issubset(set(first.keys()))MOUNT_IS_TOPVIEWRS = mount_provides_topviewrs()print("mount provides TOPVIEWRS schema:", MOUNT_IS_TOPVIEWRS)if not MOUNT_IS_TOPVIEWRS:    msg = (        "\nSTOP. The mounted dataset is not TOPVIEWRS.\n"        "  mounted annotations : %s\n"        "  mounted images      : %s\n"        "  mounted schema      : GQA visual-CoT (bboxs / full_answer / reasoning / thought)\n"        "  required schema     : %s\n"        "\nThis notebook will NOT score TOPVIEWRS on GQA records. Two ways forward:\n"        "  (a) Full reproduction: request the complete benchmark from the authors\n"        "      (cl917@cam.ac.uk, per the HF dataset card), mount it, and set\n"        "      TOPVIEWRS_LOCAL_DIR in CONFIG.\n"        "  (b) Run the next cell, which downloads the public release of chengzu/topviewrs.\n"        "      Its size is verified against the paper's 11,384 / 5,539 / 5,845 before any\n"        "      scoring, and `fidelity` in results.csv is set from the measured counts.\n"    ) % (GQA_ANNOT_PATH, GQA_IMAGE_ROOT, sorted(TOPVIEWRS_FIELDS))    print(msg)    ALLOW_PUBLIC_SUBSET = True   # set False to make this cell raise and halt the whole run    if not ALLOW_PUBLIC_SUBSET:        raise SetupError(msg)    print("ALLOW_PUBLIC_SUBSET=True -> continuing with path (b). Deviation D1 is in force.")

## LOAD DATA

In [ ]:
# ============================== LOAD DATA (1/2): snapshot + discovery ======# The dataset card documents a loading-script API:#   load_dataset(repo, trust_remote_code=True, map_type=..., task_split=..., image_save_dir=...)# `datasets` >= 4.0 removed script execution, and downgrading `datasets` drags numpy below 2.0# and kills the kernel. So this cell fetches the repo and INSPECTS it; the next cell has three# real loaders. Nothing here invents records.import datasets as hfdatasetsimport pandas as pdimport glob, tarfile, zipfile, importlib.utilfrom collections import CounterDS_MAJOR = vtuple(hfdatasets.__version__)[0]print("datasets version:", hfdatasets.__version__, "| script loading available:", DS_MAJOR < 4)SNAPSHOT_DIR = NoneFILE_INDEX: List[str] = []IMG_INDEX: Dict[str, List[str]] = {}def is_junk(path: str) -> bool:    # the released zip carries macOS resource forks and notebook checkpoints; both shadow the    # real files in a basename index and neither is data    parts = path.replace(os.sep, "/").split("/")    return any(p == "__MACOSX" or p == ".ipynb_checkpoints" or p.startswith("._") or               p == ".DS_Store" for p in parts)def build_snapshot():    global SNAPSHOT_DIR, FILE_INDEX, IMG_INDEX    if TOPVIEWRS_LOCAL_DIR and os.path.isdir(TOPVIEWRS_LOCAL_DIR):        SNAPSHOT_DIR = TOPVIEWRS_LOCAL_DIR        print("using TOPVIEWRS_LOCAL_DIR:", SNAPSHOT_DIR)    else:        from huggingface_hub import snapshot_download        print("downloading", TOPVIEWRS_HF_REPO, "(large; cached after the first run)")        SNAPSHOT_DIR = snapshot_download(repo_id=TOPVIEWRS_HF_REPO, repo_type="dataset")        print("snapshot at:", SNAPSHOT_DIR)    marker = os.path.join(IMAGE_SAVE_DIR, ".extracted")    if os.path.exists(marker):        print("archives already extracted; skipping")    else:        for arc in sorted(glob.glob(os.path.join(SNAPSHOT_DIR, "**", "*"), recursive=True)):            low = arc.lower()            if low.endswith(".zip"):                with zipfile.ZipFile(arc) as z:                    z.extractall(IMAGE_SAVE_DIR)                print("extracted", os.path.basename(arc))            elif low.endswith((".tar", ".tar.gz", ".tgz")):                with tarfile.open(arc) as t:                    t.extractall(IMAGE_SAVE_DIR)                print("extracted", os.path.basename(arc))        with open(marker, "w") as f:            f.write("ok\n")    for root in [SNAPSHOT_DIR, IMAGE_SAVE_DIR]:        for dirpath, dirnames, filenames in os.walk(root):            dirnames[:] = [d for d in dirnames                           if d not in ("__MACOSX", ".ipynb_checkpoints") and not d.startswith("._")]            for fn in filenames:                full = os.path.join(dirpath, fn)                if is_junk(full):                    continue                FILE_INDEX.append(full)                if fn.lower().endswith((".png", ".jpg", ".jpeg")):                    IMG_INDEX.setdefault(fn, []).append(full)    print("files indexed:", len(FILE_INDEX), "| unique image basenames:", len(IMG_INDEX))def resolve_map_path(p) -> Optional[str]:    if not isinstance(p, str) or not p:        return None    for c in (p.replace("<IMAGE_SAVE_DIR>", IMAGE_SAVE_DIR), p,              os.path.join(IMAGE_SAVE_DIR, p.lstrip("/")),              os.path.join(SNAPSHOT_DIR or "", p.lstrip("/"))):        if c and os.path.exists(c):            return c    hits = [h for h in IMG_INDEX.get(os.path.basename(p), []) if not is_junk(h)]    if len(hits) == 1:        return hits[0]    if len(hits) > 1:        norm = p.replace(os.sep, "/")[::-1]        hits.sort(key=lambda h: -len(os.path.commonprefix([h.replace(os.sep, "/")[::-1], norm])))        return hits[0]    return Nonebuild_snapshot()# ------------------------------- discovery ----------------------------------NON_IMAGE = [p for p in FILE_INDEX if not p.lower().endswith((".png", ".jpg", ".jpeg"))]print("\nnon-image files in the release (%d):" % len(NON_IMAGE))for p in sorted(NON_IMAGE)[:60]:    print("   ", p.replace(SNAPSHOT_DIR, "<SNAP>").replace(IMAGE_SAVE_DIR, "<IMGDIR>"))SCRIPT_CANDIDATES = sorted([p for p in NON_IMAGE if p.endswith(".py")], key=len)print("\nloading-script candidates:", [os.path.basename(p) for p in SCRIPT_CANDIDATES])if SCRIPT_CANDIDATES:    with open(SCRIPT_CANDIDATES[0]) as f:        src = f.read()    print("\n--- %s (first 200 lines) ---" % os.path.basename(SCRIPT_CANDIDATES[0]))    print("\n".join(src.splitlines()[:200]))    print("--- end ---")def describe_json(path: str, depth: int = 0, maxdepth: int = 3):    with open(path) as f:        obj = json.load(f)    def walk(o, d, prefix):        pad = "  " * d        if isinstance(o, dict):            keys = list(o.keys())            print("%s%s dict, %d keys: %s" % (pad, prefix, len(keys), keys[:12]))            if d < maxdepth and keys:                walk(o[keys[0]], d + 1, "['%s']" % keys[0])        elif isinstance(o, list):            print("%s%s list, len %d" % (pad, prefix, len(o)))            if d < maxdepth and o:                walk(o[0], d + 1, "[0]")        else:            print("%s%s %s = %r" % (pad, prefix, type(o).__name__, str(o)[:100]))    walk(obj, depth, os.path.basename(path))ANNOT_FILES = [p for p in NON_IMAGE if p.lower().endswith((".json", ".jsonl", ".parquet", ".csv"))               and not os.path.basename(p).startswith("dataset_info")]print("\nannotation-file candidates:", len(ANNOT_FILES))for p in sorted(ANNOT_FILES)[:3]:    print("\n--- structure of", p.replace(IMAGE_SAVE_DIR, "<IMGDIR>"), "---")    if p.lower().endswith(".json"):        describe_json(p)SCENES_ON_DISK = sorted({os.path.basename(os.path.dirname(os.path.dirname(p)))                         for p in FILE_INDEX if "/mp3d/" in p.replace(os.sep, "/")})PAPER_SCENES = ["17DRP5sb8fy", "2azQ1b91cZZ", "2t7WUuJeko7", "5LpN3gDmAk7",                "EU6Fwq7SyZv", "8WUmhLawc2A", "i5noydFURQK"]   # Appendix B.1print("\nscenes on disk (%d):" % len(SCENES_ON_DISK), SCENES_ON_DISK)print("scenes named in the paper (App. B.1, 7):", PAPER_SCENES)extra = [s for s in SCENES_ON_DISK if s and s not in PAPER_SCENES]if extra:    print("EXTRA scenes not named in App. B.1:", extra, "-> recorded as a deviation")

In [ ]:
# ============================== LOAD DATA (2/2): three real loaders ========REQUIRED_COLS = ["question", "choices", "labels", "map_path", "question_ability", "scene_id"]# The discovery cell printed topviewrs.py. Its record construction is:#   map_key   = "rgb" if map_type == "realistic" else map_type#   data_list = json.load(open(released_<map_type>_datasets.json))[task_split]#   map_path  = os.path.join(image_save_dir, item[f"{map_key}_map"])#   fields    = index, scene_id, question, choices, labels,#               choice_type = str(item["question_meta_data"]["choices"]),#               question_ability = item["ability"], (+ reference_path when present)# PATH B below is a line-for-line port of that, so it needs no schema guessing.MAP_KEY = {"realistic": "rgb", "semantic": "semantic"}RELEASED_JSON_NAME = "released_%s_datasets.json"# ---------------- PATH A: the documented load_dataset script API -------------def load_path_a(map_type: str, task_split: str):    if DS_MAJOR >= 4:        return None, "datasets %s removed script-based loading" % hfdatasets.__version__    try:        ds = hfdatasets.load_dataset(            TOPVIEWRS_HF_REPO, trust_remote_code=True, map_type=map_type,            task_split=task_split, image_save_dir=IMAGE_SAVE_DIR)        split = ds[list(ds.keys())[0]] if hasattr(ds, "keys") else ds        return [dict(r) for r in split], None    except Exception as e:                      # noqa: BLE001 - reported, then PATH B is tried        return None, repr(e)# ---------------- PATH B: port of the authors' _generate_examples -----------def load_path_b(map_type: str, task_split: str):    name = RELEASED_JSON_NAME % map_type    path = next((p for p in NON_IMAGE if os.path.basename(p) == name), None)    if path is None:        return None, "%s not found in the snapshot" % name    with open(path) as f:        blob = json.load(f)    if task_split not in blob:        return None, "key %r absent from %s; keys present: %s" % (task_split, name, list(blob))    map_key = MAP_KEY[map_type]    rows = []    for idx, item in enumerate(blob[task_split]):        r = {            "index": idx,            "scene_id": item["scene_id"],            "question": item["question"],            "choices": item["choices"],            "labels": item["labels"],            "choice_type": str(item["question_meta_data"]["choices"]),            "map_path": os.path.join(IMAGE_SAVE_DIR, item["%s_map" % map_key]),            "question_ability": item["ability"],        }        if "reference_path" in item:            r["reference_path"] = item["reference_path"]        rows.append(r)    return (rows, None) if rows else (None, "%s[%s] is empty" % (name, task_split))# ---------------- PATH C: import the authors' script and drive it -----------# Last resort: the authors' own code, executed around the datasets 5.x script ban. Slower,# because _split_generators re-downloads and re-unpacks data.zip._SCRIPT_MOD = Nonedef load_script_module():    global _SCRIPT_MOD    if _SCRIPT_MOD is not None:        return _SCRIPT_MOD    if not SCRIPT_CANDIDATES:        raise FileNotFoundError("no .py loading script in the snapshot")    path = SCRIPT_CANDIDATES[0]    spec = importlib.util.spec_from_file_location("topviewrs_authors_script", path)    mod = importlib.util.module_from_spec(spec)    sys.modules[spec.name] = mod    spec.loader.exec_module(mod)    _SCRIPT_MOD = mod    return moddef _builder_base():    base = getattr(hfdatasets, "GeneratorBasedBuilder", None)    if base is None:        from datasets.builder import GeneratorBasedBuilder as base  # noqa: N813    return basedef load_path_c(map_type: str, task_split: str):    try:        mod = load_script_module()        base = _builder_base()        classes = [o for o in vars(mod).values()                   if isinstance(o, type) and issubclass(o, base) and o is not base]        if not classes:            return None, "no GeneratorBasedBuilder subclass in %s" % os.path.basename(SCRIPT_CANDIDATES[0])        cls = classes[0]        b = cls(cache_dir=os.path.join(OUT_DIR, "hf_builder_cache"),                base_path=SNAPSHOT_DIR,                map_type=map_type, task_split=task_split, image_save_dir=IMAGE_SAVE_DIR)        # base_path on the DownloadManager is what resolves the script's relative        # "data.zip" / "released_*_datasets.json" against the snapshot instead of cwd        dl = hfdatasets.DownloadManager(base_path=SNAPSHOT_DIR)        rows = []        for g in b._split_generators(dl):            for _, ex in b._generate_examples(**g.gen_kwargs):                rows.append(dict(ex))        return (rows, None) if rows else (None, "authors' script yielded 0 rows")    except Exception as e:                      # noqa: BLE001 - reported; the cell then raises        return None, repr(e)# --------------------------------- run --------------------------------------DATA: Dict[Tuple[str, str], List[Dict[str, Any]]] = {}LOADER_USED: Dict[Tuple[str, str], str] = {}errors: List[str] = []for mt in MAP_TYPES:    for ts in TASK_SPLITS:        attempts = []        rows, err = load_path_a(mt, ts)        used = "A (load_dataset script API)"        if rows is None:            attempts.append("A: %s" % err)            rows, err = load_path_b(mt, ts)            used = "B (port of the authors' _generate_examples)"        if rows is None:            attempts.append("B: %s" % err)            rows, err = load_path_c(mt, ts)            used = "C (authors' script imported directly)"        if rows is None:            attempts.append("C: %s" % err)            errors.append("%s/%s -> %s" % (mt, ts, " | ".join(attempts)))            continue        missing = [c for c in REQUIRED_COLS if c not in rows[0]]        if missing:            errors.append("%s/%s loaded via %s but missing %s; present: %s"                          % (mt, ts, used, missing, sorted(rows[0].keys())))            continue        for r in rows:            r["map_path"] = resolve_map_path(r["map_path"])        DATA[(mt, ts)] = rows        LOADER_USED[(mt, ts)] = used        print("%-10s %-28s n=%5d  loader=%s" % (mt, ts, len(rows), used))if errors:    raise SetupError(        "Could not load TOPVIEWRS. Failures:\n  " + "\n  ".join(errors)        + "\n\nThe cell above printed the loading script and the annotation-file structure; "          "align PATH B with that structure. Do not proceed with substitute data.")TOTAL_LOADED = sum(len(v) for v in DATA.values())BY_MAP = {mt: sum(len(v) for (m, _), v in DATA.items() if m == mt) for mt in MAP_TYPES}SCENES_USED = sorted({str(r["scene_id"]) for v in DATA.values() for r in v})print("\nscenes referenced by loaded records (%d):" % len(SCENES_USED), SCENES_USED)print("scenes named in App. B.1 (7):", PAPER_SCENES)unused = [s_ for s_ in SCENES_ON_DISK if s_ and s_ not in SCENES_USED]beyond = [s_ for s_ in SCENES_USED if s_ not in PAPER_SCENES]print("on disk but unreferenced:", unused)print("referenced but not named in App. B.1:", beyond, "-> deviation D13" if beyond else "")print("\nTOTAL loaded:", TOTAL_LOADED, "| by map type:", BY_MAP)print("Paper (Sec. 4): 11,384 total = 5,539 realistic + 5,845 semantic.")if TOTAL_LOADED != 11384:    print("-> Not the full benchmark. Deviation D1 applies; n_eval per row is written to")    print("   results.csv from these counts, never from the paper's counts.")else:    print("-> Full benchmark size matched. D1 does not apply to corpus size.")

In [ ]:
# --- verify every map image resolves on disk; never generate a substitute ---missing_maps = []for (mt, ts), split in DATA.items():    for r in split:        p = r["map_path"]        if p is None or not os.path.exists(p):            missing_maps.append((mt, ts, p))print("map images checked :", TOTAL_LOADED)print("map images missing :", len(missing_maps))if missing_maps:    for row in missing_maps[:20]:        print("   ", row)    raise SetupError(        "%d map images referenced by the dataset do not exist under IMAGE_SAVE_DIR=%s. "        "Re-run the download; do not substitute images." % (len(missing_maps), IMAGE_SAVE_DIR)    )# --- sub-task census, analogue of paper Table 9 (counts here are OUR data, not the paper's) ---census = []for (mt, ts), split in DATA.items():    for ability, c in sorted(Counter(r["question_ability"] for r in split).items()):        census.append({"map_type": mt, "task": TASK_DISPLAY[ts], "sub_task": ability, "n": c})census_df = pd.DataFrame(census)census_df.to_csv(os.path.join(OUT_DIR, "loaded_subtask_census.csv"), index=False)display(census_df)

## PREPROCESSING

In [ ]:
# ============================= PREPROCESSING ================================from PIL import Imagefrom collections import Counterdef format_options(choices) -> str:    # Appendix C.2: 'A. bed; B. chair; C. table; D. cushion'    return "; ".join("%s. %s" % (OPTION_LETTERS[i], c) for i, c in enumerate(as_str_list(choices)))def as_str_list(x) -> List[str]:    # parquet round-trips can hand back numpy arrays of numpy str; normalise to plain python    return [str(v) for v in list(x)]def gold_index(choices, labels) -> int:    # HF card: "The label's position in the choices can be used to determine whether it is A,B,C,D."    ch, lb = as_str_list(choices), as_str_list(labels)    if len(lb) != 1:        raise ValueError("expected exactly one gold label (paper Sec. 3 fn. 4), got %r" % (lb,))    if lb[0] not in ch:        raise ValueError("gold label %r is not among the choices %r" % (lb[0], ch))    return ch.index(lb[0])_MAPPING_CACHE: Dict[str, str] = {}def semantic_mapping_block(map_path: str, include_path_symbols: bool) -> str:    # Sec. 5 (Prompts): "We only provide the color-object mappings of the colors that are    # presented in the semantic map".    # ASSUMPTION A7: presence = exact RGB match against the Table 4 palette.    # ASSUMPTION A8: lines ordered by descending pixel count.    key = map_path + ("|sym" if include_path_symbols else "")    if key in _MAPPING_CACHE:        return _MAPPING_CACHE[key]    with Image.open(map_path) as im:        rgb = im.convert("RGB")        counts = Counter(rgb.getdata())    palette = dict(RGB_TO_LABEL)    if include_path_symbols:        palette.update(PATH_SYMBOL_COLORS)    present = [(c, palette[c], n) for c, n in counts.items() if c in palette]    present.sort(key=lambda t: -t[2])    block = "\n".join("(%d, %d, %d) -> %s" % (c[0], c[1], c[2], lab) for c, lab, _ in present)    _MAPPING_CACHE[key] = block    return blockdef build_prompt(rec: Dict[str, Any], map_type: str, task_split: str, cot: bool) -> str:    q = rec["question"]    opts = format_options(rec["choices"])    dynamic = (task_split == "dynamic_spatial_reasoning")    instr = TASK_SPECIFIC_INSTRUCTION if rec["question_ability"] in INSTRUCTION_SUBTASKS else ""    if cot:        # Tables 13/14 cover Static Spatial Reasoning only.        tmpl = PROMPT_COT_REALISTIC if map_type == "realistic" else PROMPT_COT_SEMANTIC    elif map_type == "realistic":        tmpl = PROMPT_REALISTIC_DYNAMIC if dynamic else PROMPT_REALISTIC_STATIC    else:        tmpl = PROMPT_SEMANTIC_DYNAMIC if dynamic else PROMPT_SEMANTIC_STATIC    if "<MAPPING>" in tmpl:        tmpl = tmpl.replace("<MAPPING>", semantic_mapping_block(rec["map_path"], dynamic))    if "<TASK-SPECIFIC INSTRUCTION>" in tmpl:        tmpl = tmpl.replace("<TASK-SPECIFIC INSTRUCTION>", instr)    return tmpl.replace("<QUESTION>", q).replace("<OPTIONS>", opts)# smoke check on one real record from each cell of the grid (no synthetic data)for (mt, ts), split in DATA.items():    r = split[0]    p = build_prompt(r, mt, ts, cot=False)    print("=" * 70)    print(mt, "|", ts, "| ability:", r["question_ability"], "| gold idx:",          gold_index(r["choices"], r["labels"]))    print(p[:600] + ("..." if len(p) > 600 else ""))

In [ ]:
# ---------------- answer extraction and PM normalization -------------------import re, string as _string_PUNCT = str.maketrans("", "", _string.punctuation)def normalize_words(text: str) -> set:    # ASSUMPTION A6: lowercase, strip punctuation, split on whitespace and underscore.    t = text.lower().replace("_", " ").translate(_PUNCT)    return {w for w in t.split() if w}_LETTER_RE = re.compile(r"\b([ABCD])\b")_ANSWER_IS_RE = re.compile(r"the answer is\s*[:\-]?\s*\(?([ABCD])\)?", re.IGNORECASE)def extract_option_index(gen_text: str, choices: List[str]) -> int:    # ASSUMPTION A5. Returns -1 for UNPARSED. The paper reports models that ignore the    # instruction (Sec. 5.1) but does not give its parser.    t = (gen_text or "").strip()    m = _ANSWER_IS_RE.search(t)    if m:        return OPTION_LETTERS.index(m.group(1).upper())    m = _LETTER_RE.search(t)    if m:        return OPTION_LETTERS.index(m.group(1).upper())    norm = normalize_words(t)    for i, c in enumerate(choices):        if norm and norm == normalize_words(c):            return i    return -1def exact_match(pred_idx: int, gold_idx: int) -> float:    # Sec. 5: "EM measures whether the predicted option indices are exactly the same as the    # label indices."    return 1.0 if pred_idx == gold_idx else 0.0def partial_match(pred_idx: int, gold_idx: int, choices: List[str], gen_text: str) -> float:    # Sec. 5: PM = |{labels} ∩ {predictions}| / max(|{labels}|, |{predictions}|)    gold_words = normalize_words(choices[gold_idx])    pred_words = normalize_words(choices[pred_idx]) if pred_idx >= 0 else normalize_words(gen_text)    denom = max(len(gold_words), len(pred_words))    if denom == 0:        return 0.0    return len(gold_words & pred_words) / denom# sanity of the scorer itself against the paper's own worked example (Sec. 5):# gold 'top right', prediction 'top left' -> one shared word out of two._demo_choices = ["top right", "top left"]assert abs(partial_match(1, 0, _demo_choices, "") - 0.5) < 1e-9assert exact_match(1, 0) == 0.0 and exact_match(0, 0) == 1.0print("scorer sanity checks pass (structural, not a result)")

## FEATURE SELECTION**Not applicable.** The paper has no feature-selection stage: every model is a frozen, zero-shotVLM consuming a raw image plus a templated prompt (§5 *Models and Implementation*). This section ispresent only to keep your required ordering intact.

## MODEL

In [ ]:
# ================================ MODEL =====================================# Frozen zero-shot VLMs. No training, no adapters, no added modules.import gcfrom transformers import AutoProcessor, AutoTokenizer, AutoModelForCausalLMdef _max_memory():    if N_GPU == 0:        return None    # leave ~1.5 GB per card for activations and the KV cache    per = torch.cuda.get_device_properties(0).total_memory / 1e9 - 1.5    mm = {i: "%dGiB" % int(per) for i in range(N_GPU)}    mm["cpu"] = "0GiB"      # refuse CPU offload: it would silently make the run un-timeable    return mmclass VLM:    def __init__(self, name: str):        self.name = name        self.spec = MODEL_REGISTRY[name]        self.family = self.spec["family"]        self.hf_id = self.spec["hf_id"]        self.model = None        self.processor = None        self.tokenizer = None    def load(self):        kw = dict(torch_dtype=DTYPE, low_cpu_mem_usage=True)        if N_GPU > 0:            kw.update(device_map="auto", max_memory=_max_memory())        fam = self.family        if fam == "llava_next":            from transformers import LlavaNextForConditionalGeneration            self.processor = AutoProcessor.from_pretrained(self.hf_id)            self.model = LlavaNextForConditionalGeneration.from_pretrained(self.hf_id, **kw)        elif fam == "idefics":            from transformers import IdeficsForVisionText2Text            self.processor = AutoProcessor.from_pretrained(self.hf_id)            self.model = IdeficsForVisionText2Text.from_pretrained(self.hf_id, **kw)        elif fam == "xcomposer2":            self.tokenizer = AutoTokenizer.from_pretrained(self.hf_id, trust_remote_code=True)            self.model = AutoModelForCausalLM.from_pretrained(                self.hf_id, trust_remote_code=True, **kw)            self.model.tokenizer = self.tokenizer        elif fam == "qwen_vl":            self.tokenizer = AutoTokenizer.from_pretrained(self.hf_id, trust_remote_code=True)            self.model = AutoModelForCausalLM.from_pretrained(                self.hf_id, trust_remote_code=True, fp16=(DTYPE == torch.float16),                device_map="auto" if N_GPU > 0 else None)        else:            raise ValueError("unknown family %r" % fam)        self.model.eval()        if N_GPU == 0:            self.model.to("cpu")        return self    def n_params(self) -> int:        return sum(p.numel() for p in self.model.parameters())    @torch.no_grad()    def generate(self, image_path: str, prompt: str, gen_kwargs: Dict[str, Any]) -> str:        gk = {k: v for k, v in gen_kwargs.items() if v is not None}        # temperature is meaningless (and warned about) when do_sample is False        if gk.get("do_sample") is False:            gk.pop("temperature", None)            gk.pop("top_p", None)        fam = self.family        if fam == "llava_next":            image = Image.open(image_path).convert("RGB")            conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]            text = self.processor.apply_chat_template(conv, add_generation_prompt=True)            inputs = self.processor(images=image, text=text, return_tensors="pt").to(self.model.device)            out = self.model.generate(**inputs, **gk)            new = out[0][inputs["input_ids"].shape[1]:]            return self.processor.decode(new, skip_special_tokens=True).strip()        if fam == "idefics":            image = Image.open(image_path).convert("RGB")            msg = ["User:", image, prompt, "<end_of_utterance>", "\nAssistant:"]            inputs = self.processor([msg], return_tensors="pt").to(self.model.device)            bad = self.processor.tokenizer(["<image>", "<fake_token_around_image>"],                                           add_special_tokens=False).input_ids            out = self.model.generate(**inputs, bad_words_ids=bad, **gk)            new = out[0][inputs["input_ids"].shape[1]:]            return self.processor.tokenizer.decode(new, skip_special_tokens=True).strip()        if fam == "xcomposer2":            q = "<ImageHere>" + prompt            resp, _ = self.model.chat(                self.tokenizer, query=q, image=self.model.encode_img(image_path),                history=[], do_sample=gk.get("do_sample", False),                num_beams=gk.get("num_beams", 5),                max_new_tokens=gk.get("max_new_tokens", 20),                repetition_penalty=gk.get("repetition_penalty", 1.0))            return str(resp).strip()        if fam == "qwen_vl":            query = self.tokenizer.from_list_format([                {"image": image_path}, {"text": prompt}])            resp, _ = self.model.chat(self.tokenizer, query=query, history=None)            return str(resp).strip()        raise ValueError("unknown family %r" % fam)def free(vlm: Optional[VLM]):    if vlm is not None:        del vlm.model    gc.collect()    if torch.cuda.is_available():        torch.cuda.empty_cache()print("Model wrappers defined. Dropped from the paper's roster on this hardware:")for k, v in MODELS_DROPPED_VRAM.items():    print("  VRAM  :", k, "->", v)for k, v in MODELS_DROPPED_API.items():    print("  API   :", k, "->", v)

## TRAIN

In [ ]:
# ================================ TRAIN =====================================# The paper trains nothing. Sec. 5: models are evaluated "in a zero-shot inference setup".# There is no loss, optimizer, learning-rate schedule, or epoch count to reproduce# (all NOT SPECIFIED, because none exists).TRAIN_TIME_S = 0.0        # measured: no optimizer step was executed in this notebookTRAINING_STEPS = 0        # inference-only runprint("No training stage. TRAIN_TIME_S=%.1f, TRAINING_STEPS=%d" % (TRAIN_TIME_S, TRAINING_STEPS))

## EVALUATE

In [ ]:
# =============================== EVALUATE ===================================# Runs the model on every eval record and compares its generated text to the gold option.# No sampled correctness flags anywhere: every score comes from a real generation.import time, jsonfrom sklearn.metrics import precision_recall_fscore_support, confusion_matrixdef cache_path(model_name, map_type, task_split, cot):    tag = "cot" if cot else "std"    return os.path.join(CACHE_DIR, "%s__%s__%s__%s.jsonl" % (model_name, map_type, task_split, tag))def load_cache(path):    done = {}    if os.path.exists(path):        with open(path) as f:            for line in f:                line = line.strip()                if line:                    d = json.loads(line)                    done[d["row"]] = d    return donedef run_split(vlm: VLM, map_type: str, task_split: str, cot: bool = False) -> List[Dict[str, Any]]:    split = DATA[(map_type, task_split)]    n = len(split) if MAX_ITEMS_PER_TASK is None else min(MAX_ITEMS_PER_TASK, len(split))    cpath = cache_path(vlm.name, map_type, task_split, cot)    done = load_cache(cpath)    gen_kwargs = dict(vlm.spec["gen"])    if cot:        gen_kwargs["max_new_tokens"] = COT_MAX_NEW_TOKENS   # Table 10 CoT models used 1024    t0 = time.time()    fh = open(cpath, "a")    try:        for i in range(n):            if i in done:                continue            rec = split[i]            prompt = build_prompt(rec, map_type, task_split, cot=cot)            gen = vlm.generate(rec["map_path"], prompt, gen_kwargs)   # raises on OOM; no fallback            row = {                "row": i,                "scene_id": rec["scene_id"],                "question_ability": rec["question_ability"],                "choices": as_str_list(rec["choices"]),                "gold_idx": gold_index(rec["choices"], rec["labels"]),                "gen": gen,            }            fh.write(json.dumps(row) + "\n")            fh.flush()            done[i] = row            if (i + 1) % 50 == 0:                el = time.time() - t0                print("  %s %s/%s %d/%d  %.2fs/item" % (vlm.name, map_type, task_split, i + 1, n, el / (i + 1)))    finally:        fh.close()    return [done[i] for i in range(n) if i in done]def score_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:    ems, pms, y_true, y_pred = [], [], [], []    n_unparsed = 0    for r in rows:        pi = extract_option_index(r["gen"], r["choices"])        if pi < 0:            n_unparsed += 1        ems.append(exact_match(pi, r["gold_idx"]))        pms.append(partial_match(pi, r["gold_idx"], r["choices"], r["gen"]))        y_true.append(r["gold_idx"])        y_pred.append(pi)    n = len(rows)    if n == 0:        raise ValueError("no rows to score; refusing to emit a metric for an empty set")    out = {        "n": n,        "EM": 100.0 * sum(ems) / n,      # paper reports EM/PM as percentages (Table 1)        "PM": 100.0 * sum(pms) / n,        "unparsed": n_unparsed,    }    labels4 = [0, 1, 2, 3]    try:        p, r_, f1, _ = precision_recall_fscore_support(            y_true, y_pred, labels=labels4, average="macro", zero_division=0)        out["Prec"], out["Recall"], out["F1"] = 100.0 * p, 100.0 * r_, 100.0 * f1    except ValueError as e:        print("precision_recall_fscore_support failed:", repr(e))        out["Prec"] = out["Recall"] = out["F1"] = float("nan")    try:        cm = confusion_matrix(y_true, y_pred, labels=labels4)        fprs, fnrs = [], []        total = cm.sum()        for k in range(4):            tp = cm[k, k]            fn = cm[k, :].sum() - tp            fp = cm[:, k].sum() - tp            tn = total - tp - fn - fp            fprs.append(fp / (fp + tn) if (fp + tn) > 0 else float("nan"))            fnrs.append(fn / (fn + tp) if (fn + tp) > 0 else float("nan"))        out["FPR"] = 100.0 * float(np.nanmean(fprs))        out["FNR"] = 100.0 * float(np.nanmean(fnrs))    except ValueError as e:        print("confusion_matrix failed:", repr(e))        out["FPR"] = out["FNR"] = float("nan")    return outprint("Evaluation functions defined.")print("Prec/Recall/F1: macro over the 4 letter classes A-D, one-vs-rest, zero_division=0.")print("  No positive class: the task is 4-way multiclass. The paper specifies neither the")print("  averaging nor these metrics - it reports EM and PM only (Sec. 5). See A9.")print("ROC-AUC / PR-AUC: N/A. Table 10 decoding is greedy free-form generation with")print("  max_new_tokens=20; there is no probability score independent of the label.")

In [ ]:
# ------------------------------- main loop ---------------------------------main_rows, subtask_rows, cot_rows = [], [], []for model_name in MODELS_TO_RUN:    if model_name not in MODEL_REGISTRY:        raise KeyError("%s is not in MODEL_REGISTRY" % model_name)    if torch.cuda.is_available():        torch.cuda.reset_peak_memory_stats()    print("\n" + "#" * 78)    print("# LOADING", model_name, MODEL_REGISTRY[model_name]["hf_id"])    print("#" * 78)    t_load = time.time()    vlm = VLM(model_name).load()          # raises on OOM / missing weights; no fallback path    n_params = vlm.n_params()    print("params: %d (%.2f B)  load: %.1fs" % (n_params, n_params / 1e9, time.time() - t_load))    infer_t0 = time.time()    for map_type in MAP_TYPES:        for task_split in TASK_SPLITS:            print("\n>>", model_name, map_type, task_split)            rows = run_split(vlm, map_type, task_split, cot=False)            m = score_rows(rows)            main_rows.append({"model": model_name, "map_type": map_type,                              "task": TASK_DISPLAY[task_split], **m,                              "n_params": n_params})            print("   scored n=%d unparsed=%d" % (m["n"], m["unparsed"]))            by_ability = {}            for r in rows:                by_ability.setdefault(r["question_ability"], []).append(r)            for ability, rs in sorted(by_ability.items()):                sm = score_rows(rs)                subtask_rows.append({"model": model_name, "map_type": map_type,                                     "task": TASK_DISPLAY[task_split],                                     "sub_task": ability, **sm})    if RUN_COT:        # Deviation D9: the paper ran this on GPT-4V and Gemini only.        for map_type in MAP_TYPES:            print("\n>> [CoT]", model_name, map_type, COT_TASK)            rows = run_split(vlm, map_type, COT_TASK, cot=True)            m = score_rows(rows)            cot_rows.append({"model": model_name, "map_type": map_type,                             "task": TASK_DISPLAY[COT_TASK], "cot": True, **m})    INFER_TIME_S = time.time() - infer_t0    PEAK_GPU_GB = (max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9                   if N_GPU > 0 else float("nan"))    for r in main_rows:        if r["model"] == model_name:            r["infer_time_s"] = INFER_TIME_S            r["peak_gpu_mem_gb"] = PEAK_GPU_GB    print("\n%s: inference %.1fs, peak GPU %.2f GB" % (model_name, INFER_TIME_S, PEAK_GPU_GB))    free(vlm)

## SAVE RESULTS

In [ ]:
# ============================= SAVE RESULTS =================================import pandas as pdCOLS = ["Acc", "Prec", "Recall", "F1", "ROC-AUC", "PR-AUC", "FPR", "FNR",        "Train Time", "Params", "Comm Cost", "Training Steps",        "n_eval", "source", "split", "model", "inputs", "fidelity"]EXTRA = ["PM", "Unparsed", "Infer Time", "Peak GPU Mem GB"]INPUTS_STR = "%s ; %s ; %s" % (GQA_ANNOT_PATH, GQA_IMAGE_ROOT,                               TOPVIEWRS_LOCAL_DIR or ("hf:" + TOPVIEWRS_HF_REPO))FIDELITY = "full reproduction" if (MOUNT_IS_TOPVIEWRS and TOTAL_LOADED == 11384                                   and MAX_ITEMS_PER_TASK is None) else "reduced"def fmt(x):    return "N/A" if (isinstance(x, float) and math.isnan(x)) else xrecords = []for r in main_rows:    records.append({        "Acc": r["EM"],                       # paper's headline metric is EM (Sec. 5)        "Prec": fmt(r["Prec"]), "Recall": fmt(r["Recall"]), "F1": fmt(r["F1"]),        "ROC-AUC": "N/A",                     # no label-independent score exists (see EVALUATE)        "PR-AUC": "N/A",        "FPR": fmt(r["FPR"]), "FNR": fmt(r["FNR"]),        "Train Time": TRAIN_TIME_S,        "Params": r["n_params"],        "Comm Cost": "N/A",                   # the paper measures no communication cost        "Training Steps": TRAINING_STEPS,        "n_eval": r["n"],        "source": "TopViewRS (Li et al., 2024; arXiv:2406.02537v1)",        "split": "%s | %s" % (r["map_type"], r["task"]),        "model": "%s (%s)" % (r["model"], MODEL_REGISTRY[r["model"]]["hf_id"]),        "inputs": INPUTS_STR,        "fidelity": FIDELITY,        "PM": r["PM"],        "Unparsed": r["unparsed"],        "Infer Time": r.get("infer_time_s", float("nan")),        "Peak GPU Mem GB": fmt(r.get("peak_gpu_mem_gb", float("nan"))),    })if not records:    raise RuntimeError("No rows were produced. Nothing is written; an empty results.csv would "                       "be indistinguishable from a run that scored zero.")results = pd.DataFrame(records)[COLS + EXTRA]results.to_csv(RESULTS_CSV, index=False)display(results)print("wrote", RESULTS_CSV)if subtask_rows:    st = pd.DataFrame(subtask_rows)    st.to_csv(SUBTASK_CSV, index=False)    display(st)    print("wrote", SUBTASK_CSV, "(analogue of paper Table 15 / Fig. 3)")if cot_rows:    ct = pd.DataFrame(cot_rows)    ct.to_csv(COT_CSV, index=False)    display(ct)    print("wrote", COT_CSV, "(analogue of paper Table 3; deviation D9 applies)")

In [ ]:
# --------------- runtime deviation record, written from measured state -----dev = [    {"item": "Eval corpus size", "paper": "11,384 MCQ (Sec. 4)",     "this_run": TOTAL_LOADED, "effect": "subset scores are not the paper's scores"},    {"item": "Records per task cap", "paper": "no cap",     "this_run": MAX_ITEMS_PER_TASK if MAX_ITEMS_PER_TASK is not None else "none",     "effect": "further reduction of n_eval if set"},    {"item": "Models evaluated", "paper": "10 (Sec. 5)",     "this_run": ", ".join(MODELS_TO_RUN),     "effect": "no Idefics-80B / LLaVANext-34B / GPT-4V / Gemini rows"},    {"item": "Precision", "paper": "NOT SPECIFIED", "this_run": str(DTYPE),     "effect": "small numeric drift"},    {"item": "Harness", "paper": "VLMEvalKit (Sec. 5)", "this_run": "plain transformers loop",     "effect": "answer-extraction heuristics differ (A5)"},    {"item": "Human evaluation", "paper": "4 annotators, 60 items (Sec. 5.2)",     "this_run": "not run", "effect": "Table 2 not reproduced"},    {"item": "CoT ablation", "paper": "GPT-4V + Gemini (Table 3)",     "this_run": "enabled" if RUN_COT else "not run",     "effect": "Table 3 not comparable"},    {"item": "Mounted GQA data", "paper": "n/a", "this_run": json.dumps(gqa_audit),     "effect": "audited then rejected; contributed 0 records"},    {"item": "Library versions", "paper": "VLMEvalKit, versions NOT SPECIFIED",     "this_run": json.dumps(ENV_VERSIONS),     "effect": "image preprocessing differences can shift scores"},    {"item": "Scenes referenced by loaded records", "paper": "7 (App. B.1)",     "this_run": json.dumps(SCENES_USED),     "effect": "scenes beyond the 7 named in App. B.1 shift the sub-task mix"},    {"item": "Scenes present on disk", "paper": "7 (App. B.1)",     "this_run": json.dumps(SCENES_ON_DISK),     "effect": "image-only extras; no effect unless referenced"},    {"item": "Dataset loader used", "paper": "n/a",     "this_run": json.dumps({"%s/%s" % k: v for k, v in LOADER_USED.items()}),     "effect": "both paths read the authors' files; recorded for traceability"},]dev_df = pd.DataFrame(dev)dev_df.to_csv(os.path.join(OUT_DIR, "deviations_runtime.csv"), index=False)display(dev_df)

---## What is deliberately absent- **No metric values anywhere in this notebook.** Every number in `results.csv` is produced by  `score_rows`, which reads real generations from real map images. Nothing is copied from  Table 1, Table 3, Table 9 or Table 15 into an output.- **No dummy data.** There is no `torch.randn`, no `np.random.choice` standing in for a model  output, no synthetic image, no demo mode. Missing files raise `SetupError`.- **No fill values.** `metrics` dicts are built only from computed quantities; failed sklearn  calls become `float("nan")` with the exception printed, and `nan` is rendered as `N/A` only at  write time.- **No unspecified extras.** No augmentation, no self-consistency, no option-logit rescoring, no  prompt tuning — the paper describes none of these, and adding them would change what is measured.## Where to resumeSet `MODELS_TO_RUN` to the next model and re-run from **MODEL** onwards. Prediction caches under`/kaggle/working/pred_cache/` make each split resumable at the record level, so a run killed by the12-hour limit continues where it stopped.